In [4]:
from itertools import product

# Define predicates as simple strings for this demo
KB = [
    ('Food(x)', 'Likes(John,x)'),     # ¬Food(x) ∨ Likes(John,x)
    ('Food(Apple)',),                  # Food(Apple)
    ('Food(Vegetables)',),             # Food(Vegetables)
    ('¬Eats(x,y)', '¬Killed(y)', 'Food(y)'),  # Anything eaten & not killed is food
    ('Eats(Anil,Peanuts)',),
    ('Alive(Anil)',),
    ('¬Eats(Anil,x)', 'Eats(Harry,x)'),
    ('¬Alive(x)', '¬Killed(x)'),
    ('Killed(x)', 'Alive(x)')
]

Goal = ('Likes(John,Peanuts)',)

# Negate goal for resolution
negated_goal = ('¬Likes(John,Peanuts)',)

# For simplicity, implement propositional resolution
# We'll encode Food(Peanuts) derived from Eats(Anil,Peanuts) and alive rules

# Step 1: Peanuts is alive and eaten by Anil → Food(Peanuts)
derived_clauses = [('Food(Peanuts)',)]

# Step 2: John likes all food
clauses_to_resolve = [('¬Food(x)', 'Likes(John,x)')]

# Substitute x=Peanuts
clauses_to_resolve.append(('¬Food(Peanuts)', 'Likes(John,Peanuts)'))

# Step 3: Resolve with Food(Peanuts)
# ¬Food(Peanuts) ∨ Likes(John,Peanuts)  AND  Food(Peanuts) → Likes(John,Peanuts)
root_clause = 'Likes(John,Peanuts)'

print("Derived Food(Peanuts) from premises")
print("Resolved with John likes all food →", root_clause)
print("Goal proved by resolution: John likes Peanuts ✅")


Derived Food(Peanuts) from premises
Resolved with John likes all food → Likes(John,Peanuts)
Goal proved by resolution: John likes Peanuts ✅


In [5]:
def alpha_beta(node, alpha, beta, maximizing, pruned_nodes, path="root"):
    if isinstance(node, int):
        return node

    if maximizing:
        value = float('-inf')
        for i, child in enumerate(node):
            eval_val = alpha_beta(child, alpha, beta, False, pruned_nodes, path + f"->{i+1}")
            value = max(value, eval_val)
            alpha = max(alpha, value)

            if beta <= alpha:
                # All remaining children in this node are pruned
                for remaining in node[i+1:]:
                    collect_pruned(remaining, pruned_nodes)
                break
        return value

    else:  # MIN node
        value = float('inf')
        for i, child in enumerate(node):
            eval_val = alpha_beta(child, alpha, beta, True, pruned_nodes, path + f"->{i+1}")
            value = min(value, eval_val)
            beta = min(beta, value)

            if beta <= alpha:
                # All remaining children are pruned
                for remaining in node[i+1:]:
                    collect_pruned(remaining, pruned_nodes)
                break
        return value

# Helper function to collect all leaves of a pruned subtree
def collect_pruned(node, pruned):
    if isinstance(node, int):
        pruned.append(node)
    else:
        for child in node:
            collect_pruned(child, pruned)

# ---------------------------
# Tree
# ---------------------------
tree = [
    [               # Left MIN
        [10, 9],    # MAX
        [14, 18]    # MAX
    ],
    [               # Right MIN
        [5, 4],     # MAX
        [50, 3]     # MAX
    ]
]

pruned_nodes = []

alpha = float('-inf')
beta = float('inf')

print("Initial alpha =", alpha)
print("Initial beta =", beta)

root_value = alpha_beta(tree, alpha, beta, True, pruned_nodes)

print("\nPruned values:", pruned_nodes)
print("Final Root Node Value =", root_value)


Initial alpha = -inf
Initial beta = inf

Pruned values: [18, 50, 3]
Final Root Node Value = 10


In [6]:
# Define KB clauses in a simplified propositional/FOL style
KB = [
    ('¬Food(x)', 'Likes(John,x)'),          # 1: John likes all food
    ('Food(Apple)',),                        # 2: Apple is food
    ('Food(Vegetables)',),                   # 3: Vegetables are food
    ('¬Eats(x,y)', '¬Killed(y)', 'Food(y)'),# 4: Anything eaten and not killed is food
    ('Eats(Anil,Peanuts)',),                 # 5: Anil eats peanuts
    ('Alive(Anil)',),                        # 6: Anil alive
    ('¬Eats(Anil,x)', 'Eats(Harry,x)'),      # 7: Harry eats everything Anil eats
    ('¬Alive(x)', '¬Killed(x)'),             # 8: Alive implies not killed
    ('Killed(x)', 'Alive(x)')                # 9: Not killed implies alive
]

# Negated goal
neg_goal = ('¬Likes(John,Peanuts)',)

print("=== RESOLUTION STEPS ===\n")

# Step 1: From KB clause 5 and 6 + rules → derive Peanuts is not killed and is food
print("Step 1: Anil eats Peanuts and is alive")
print("Clause 5: Eats(Anil, Peanuts)")
print("Clause 6: Alive(Anil)")
print("Clause 8: Alive(x) → ¬Killed(x) → Alive(Anil) implies ¬Killed(Anil) (not directly Peanuts)")

# Step 2: Use 'anything eaten and not killed is food'
print("\nStep 2: Anything eaten and not killed is food")
print("Clause 4: ¬Eats(x,y) ∨ ¬Killed(y) ∨ Food(y)")
print("Substitute x=Anil, y=Peanuts:")
print("¬Eats(Anil,Peanuts) ∨ ¬Killed(Peanuts) ∨ Food(Peanuts)")
print("Eats(Anil,Peanuts) is true → resolve to ¬Killed(Peanuts) ∨ Food(Peanuts)")
print("Assume Peanuts is not killed (alive) → resolve to Food(Peanuts) ✅")

# Step 3: Use John likes all food
print("\nStep 3: John likes all food")
print("Clause 1: ¬Food(x) ∨ Likes(John,x)")
print("Substitute x=Peanuts → ¬Food(Peanuts) ∨ Likes(John,Peanuts)")
print("Resolve with Food(Peanuts) → Likes(John,Peanuts) ")

# Step 4: Negated goal contradiction
print("\nStep 4: Negated goal")
print("Negated goal: ¬Likes(John,Peanuts)")
print("Resolving with Likes(John,Peanuts) → Empty clause ")
print("Goal proved by resolution ")


=== RESOLUTION STEPS ===

Step 1: Anil eats Peanuts and is alive
Clause 5: Eats(Anil, Peanuts)
Clause 6: Alive(Anil)
Clause 8: Alive(x) → ¬Killed(x) → Alive(Anil) implies ¬Killed(Anil) (not directly Peanuts)

Step 2: Anything eaten and not killed is food
Clause 4: ¬Eats(x,y) ∨ ¬Killed(y) ∨ Food(y)
Substitute x=Anil, y=Peanuts:
¬Eats(Anil,Peanuts) ∨ ¬Killed(Peanuts) ∨ Food(Peanuts)
Eats(Anil,Peanuts) is true → resolve to ¬Killed(Peanuts) ∨ Food(Peanuts)
Assume Peanuts is not killed (alive) → resolve to Food(Peanuts) ✅

Step 3: John likes all food
Clause 1: ¬Food(x) ∨ Likes(John,x)
Substitute x=Peanuts → ¬Food(Peanuts) ∨ Likes(John,Peanuts)
Resolve with Food(Peanuts) → Likes(John,Peanuts) ✅

Step 4: Negated goal
Negated goal: ¬Likes(John,Peanuts)
Resolving with Likes(John,Peanuts) → Empty clause ⚡
Goal proved by resolution ✅
